# DEV VERIFICATION COPY (not for handover)
Same as launcher/HIT401_Platform_Colab.ipynb but mounts the project Drive and uses Drive paths.


# Wildlife Image Authenticity Analysis Platform: Colab launcher
HIT401 research prototype (Prototype v2). It runs two **frozen** research detectors (the original OmniAID v2 and the
project's wildlife-adapted OmniAID v2) on one or many wildlife images. Nothing is trained or re-tuned.

**Before you start:** *Runtime -> Change runtime type -> T4 GPU*, then run the cells from top to bottom.

You need two things:
1. **The platform package**: `HIT401_Wildlife_Authenticity_Platform.zip`.
2. **The project's adapted parameter file**: `adapted_epoch_1.pt` (6.4 MB). It is *not* inside the ZIP (see
   `models/README_MODEL_SETUP.md`); the project team provides it to you privately.

The official base checkpoint (3.24 GB) is downloaded automatically from Hugging Face; no account or token is needed.

## 1. Check the GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; assert torch.cuda.is_available(), "No GPU: choose Runtime -> Change runtime type -> T4 GPU, then run again."
print("GPU:", torch.cuda.get_device_name(0))

## 2. Get the platform package
Leave `PACKAGE_PATH` empty to upload `HIT401_Wildlife_Authenticity_Platform.zip` from your computer (an upload button
appears). If the ZIP or an unzipped platform folder is in your Google Drive, put its path here instead (this needs the
Drive to be mounted: run `from google.colab import drive; drive.mount('/content/drive')` first).

In [ ]:
PACKAGE_PATH = "/content/drive/MyDrive/HIT401_Wildlife_Deepfake/FINAL_CAPSTONE_WORKSPACE/05b_Prototype_v2_Platform"   # e.g. "/content/drive/MyDrive/HIT401_Wildlife_Authenticity_Platform.zip" or a folder path; empty = upload

import os, shutil, zipfile
APP_DIR = "/content/HIT401_Wildlife_Authenticity_Platform"
def _install(src):
    if os.path.isdir(src):
        shutil.copytree(src, APP_DIR, dirs_exist_ok=True, ignore=shutil.ignore_patterns("_omniaid_repo", "_run", "__pycache__", "*.pt"))
    else:
        with zipfile.ZipFile(src) as z:
            z.extractall("/content")
if not os.path.exists(f"{APP_DIR}/app.py"):
    if PACKAGE_PATH:
        _install(PACKAGE_PATH)
    else:
        from google.colab import files
        up = files.upload()
        _install(next(iter(up)))
assert os.path.exists(f"{APP_DIR}/app.py"), "Platform files not found"
print("Platform files ready in", APP_DIR)

## 3. Install the pinned packages (about 1-2 minutes)

In [ ]:
!pip install -q -r "$APP_DIR/requirements.txt"

## 4. Provide the adapted parameter file `adapted_epoch_1.pt`
Leave `ADAPTED_PATH` empty to upload the file from your computer, or give its path in your mounted Google Drive.
The file is checked against its frozen SHA-256 (`733ab455...`); a wrong or modified file is rejected.

In [ ]:
ADAPTED_PATH = "/content/drive/MyDrive/HIT401_Wildlife_Deepfake/metadata/model_adaptation/adapted_checkpoints/adapted_epoch_1.pt"   # e.g. "/content/drive/MyDrive/adapted_epoch_1.pt"; empty = upload

import hashlib
target = f"{APP_DIR}/models/adapted_epoch_1.pt"
if not os.path.exists(target):
    if ADAPTED_PATH:
        shutil.copy(ADAPTED_PATH, target)
    else:
        from google.colab import files
        up = files.upload()
        shutil.move(next(iter(up)), target)
sha = hashlib.sha256(open(target, "rb").read()).hexdigest()
assert sha == "733ab455a3d1f677451c9368846eb44e36236d492571d7960d075da641a3cd66", f"Wrong adapted file (SHA-256 {sha})"
print("adapted_epoch_1.pt OK:", sha)

## 5. Start the platform
First start: clones the official OmniAID code (pinned commit), downloads the 3.24 GB base checkpoint, checks all
SHA-256 hashes and re-scores the 12 bundled verification images against their saved research scores (about 5-10
minutes). It will not start if anything differs. Then open the printed **App URL** in a new browser tab.
If Colab asks whether this notebook may use a secret such as `HF_TOKEN`, you may decline: no token is needed.

In [ ]:
import os
from google.colab.output import eval_js
os.environ["HIT401_ROOT_URL"] = eval_js("google.colab.kernel.proxyPort(7860)").rstrip("/")
os.environ["HIT401_APP_DIR"] = APP_DIR
%cd $APP_DIR
print("App URL (open in a new tab once the app has started):", os.environ["HIT401_ROOT_URL"])
%run app.py

## 6. Optional: record verification evidence
Re-checks model integrity, re-scores the bundled verification images, tests a mixed batch with invalid files and writes
`verification/results/v2_verification.json` and `v2_verification_summary.md` inside the platform folder.

In [ ]:
%run -i $APP_DIR/verification/verify_v2.py